# 04 Neural network

A small multilayer perceptron (MLP) with the same split and the same 3 features as logistic regression, so the comparison is fair.
We tune it on the validation students only. The test set is not used here.


In [10]:
import sys
sys.path.append("..")  

import numpy as np
import pandas as pd
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, accuracy_score, log_loss

from src.data import load_data, split_students
from src.features import build_features

# 1. Load the data, split the students, add the clues
df = build_features(split_students(load_data()))

# 2. The same 3 features as logistic regression (picked by forward selection in notebook 03)
features = ["acc_prev", "q_correct_rate", "acc_prev_L2"]

# 3. Training and validation students (test is only used in notebook 05)
train = df[df["split"] == "train"]
val = df[df["split"] == "val"]

print("Training answers:  ", f"{len(train):,}")
print("Validation answers:", f"{len(val):,}")

Training answers:   977,349
Validation answers: 200,546


## 1. A first neural network
Two hidden layers (32 and 16 neurons). Early stopping ends training when it stops improving.

In [ ]:
# ---------- A helper: build a neural network ----------
def make_mlp(hidden, lr):
    network = MLPClassifier(hidden_layer_sizes=hidden, 
                            learning_rate_init=lr,  
                            early_stopping=True, 
                            batch_size=2048,
                            random_state=42)
    return make_pipeline(StandardScaler(), network)


# ---------- A helper: scores on validation ----------
def val_scores(model):
    prob = model.predict_proba(val[features])[:, 1]
    y = val["IsCorrect"]
    return {"val AUC": round(roc_auc_score(y, prob), 4),
            "val accuracy": round(accuracy_score(y, prob >= 0.5), 4),
            "val log loss": round(log_loss(y, prob), 4)}


# ---------- First try: 2 hidden layers with 32 and 16 neurons ----------
mlp = make_mlp(hidden=(32, 16), lr=0.001)
mlp.fit(train[features], train["IsCorrect"])

print("Epochs trained:", mlp[-1].n_iter_)
pd.DataFrame([val_scores(mlp)])

Epochs trained: 15


,val AUC,val accuracy,val log loss
0,0.7783,0.7112,0.5588


**Takeaway:** A first network with two hidden layers (32 and 16 neurons) trains for 15 epochs before early stopping ends it. It already does a little better than logistic regression: AUC 0.7783 vs. 0.7759.

## 2. Tuning
We try a few network sizes and learning rates, and pick the one with the best validation log loss (the same rule as logistic regression).

In [12]:
# ---------- Try every combination of size and learning rate ----------
sizes = [(16,), (32, 16), (64, 32), (64, 32, 16)]     # network sizes (neurons per layer)
learning_rates = [0.001, 0.01]

rows = []
for hidden in sizes:
    for lr in learning_rates:
        model = make_mlp(hidden, lr)
        model.fit(train[features], train["IsCorrect"])

        scores = val_scores(model)                     # AUC, accuracy, log loss on validation
        rows.append({"hidden": hidden,
                     "lr": lr,
                     "epochs": model[-1].n_iter_,
                     "val AUC": scores["val AUC"],
                     "val accuracy": scores["val accuracy"],
                     "val log loss": scores["val log loss"]})

tuning = pd.DataFrame(rows).sort_values("val log loss")   # best (lowest log loss) on top

# ---------- Pick the best one ----------
best_hidden = tuning.iloc[0]["hidden"]
best_lr = tuning.iloc[0]["lr"]
print("Chosen:", best_hidden, "with learning rate", best_lr)
tuning

Chosen: (64, 32) with learning rate 0.001


,hidden,lr,epochs,val AUC,val accuracy,val log loss
4,"(64, 32)",0.001,31,0.7793,0.7114,0.5568
6,"(64, 32, 16)",0.001,32,0.7795,0.7113,0.5568
3,"(32, 16)",0.010,22,0.7792,0.7116,0.5571
1,"(16,)",0.010,22,0.7791,0.7111,0.5573
7,"(64, 32, 16)",0.010,20,0.7792,0.7113,0.5573
0,"(16,)",0.001,18,0.7788,0.7115,0.5578
5,"(64, 32)",0.010,20,0.7790,0.7114,0.5578
2,"(32, 16)",0.001,15,0.7783,0.7112,0.5588


**Takeaway:** All 8 networks are almost equally good (AUC 0.778–0.780, log loss 0.557–0.559), and all of them beat logistic regression (AUC 0.7759). The size and learning rate barely matter. Two networks tie for the best log loss (0.5568), and we choose the simpler one: (64, 32) with learning rate 0.001.

## 3. Final neural network vs. logistic regression

In [ ]:
# ---------- 1. Train the final network with the best settings ----------
final_mlp = make_mlp(best_hidden, best_lr)
final_mlp.fit(train[features], train["IsCorrect"])
nn_prob = final_mlp.predict_proba(val[features])[:, 1]

# ---------- 2. Load logistic regression's saved validation predictions ----------
lr_preds = pd.read_csv("../results/preds_logreg.csv")
lr_val = val[["AnswerId", "IsCorrect"]].merge(lr_preds, on="AnswerId")   # match by answer
lr_prob = lr_val["p_correct"]

# ---------- 3. Side-by-side table ----------
y = val["IsCorrect"]
comparison = pd.DataFrame({
    "model":        ["Logistic regression", f"Neural network {best_hidden}"],
    "val AUC":      [roc_auc_score(lr_val["IsCorrect"], lr_prob), roc_auc_score(y, nn_prob)],
    "val accuracy": [accuracy_score(lr_val["IsCorrect"], lr_prob >= 0.5), accuracy_score(y, nn_prob >= 0.5)],
    "val log loss": [log_loss(lr_val["IsCorrect"], lr_prob), log_loss(y, nn_prob)],
}).round(4)
comparison

,model,val AUC,val accuracy,val log loss
0,Logistic regression,0.7759,0.7085,0.5675
1,"Neural network (64, 32, 16)",0.7795,0.7116,0.5568


**Takeaway:** The neural network is better on every score, but only by a little: AUC 0.7795 vs. 0.7759, 71.2% vs. 70.9% right, and a lower log loss (0.5568 vs. 0.5675). Most of the gain comes from the features, not from the model, and logistic regression is easier to explain.

## 4. Save predictions
Same format as logistic regression. The test set is not scored here.

In [6]:
to_predict = df[df["split"].isin(["val", "test"])]
preds = pd.DataFrame({
    "AnswerId": to_predict["AnswerId"],
    "split": to_predict["split"],
    "p_correct": final_mlp.predict_proba(to_predict[features])[:, 1],
})
preds.to_csv("../results/preds_nn.csv", index=False)
print(preds.shape)

(405378, 3)


**Takeaway:** We saved 405,378 validation and test predictions to `results/preds_nn.csv` for notebook 05.